In [ ]:
import numpy as np


import torch
import torch.nn as nn

In [ ]:
with open("./datasets/1268-0.txt", "r", encoding="utf-8") as fp:
    text = fp.read()

start_idx = text.find("THE MYSTERIOUS ISLAND")
end_indx = text.find("End of the Project Gutenberg")
text = text[start_idx:end_indx]

char_set = set(text)

In [6]:
print("Total Length:", len(text))
print("Unique Characters:", len(char_set))

Total Length: 1112310
Unique Characters: 80


In [ ]:
chars_sorted = sorted(char_set)
char2int = {ch: i for i, ch in enumerate(chars_sorted)}
char_array = np.array(chars_sorted)

text_encoded = np.array([char2int[ch] for ch in text], dtype=np.int32)
print(f"Text encoded shape: {text_encoded.shape}")

Text encoded shape: (1112310,)


In [10]:
print(text[:15], "== Encoding ==>", text_encoded[:15])
print(text_encoded[15:21], "== Decoding ==>", text[15:21])

THE MYSTERIOUS  == Encoding ==> [44 32 29  1 37 48 43 44 29 42 33 39 45 43  1]
[33 43 36 25 38 28] == Decoding ==> ISLAND


In [11]:
for ex in text_encoded[:10]:
    print(f"{ex} -> {char_array[ex]}")

44 -> T
32 -> H
29 -> E
1 ->  
37 -> M
48 -> Y
43 -> S
44 -> T
29 -> E
42 -> R


In [ ]:
from torch.utils.data import Dataset

SEQ_LENGTH = 40
CHUNK_SIZE = SEQ_LENGTH + 1
text_chunks = [
    text_encoded[i : i + CHUNK_SIZE] for i in range(len(text_encoded) - CHUNK_SIZE)
]


class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks

    def __len__(self):
        return len(self.text_chunks)

    def __getitem__(self, idx):
        text_chunk = self.text_chunks[idx]
        return text_chunk[:-1].long(), text_chunk[1:].long()


seq_dataset = TextDataset(torch.tensor(text_chunks))

/tmp/ipykernel_224374/1569111636.py:23: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:274.)
  seq_dataset = TextDataset(torch.tensor(text_chunks))


In [14]:
# for i, (seq, target) in enumerate(seq_dataset):
#     print(f"Input (x): {repr(''.join(char_array[seq]))}")
#     print(f"Target (y): {repr(''.join(char_array[target]))}")
#     print()

#     if i == 2:
#         break

In [28]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64
torch.manual_seed(1)
seq_dl = DataLoader(seq_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

---
## Building a character-level RNN model

In [42]:
device = "cuda"

In [43]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)

    def forward(self, x, hidden, cell):
        out = self.embedding(x).unsueeze(1)
        out, (hidden, cell) = self.rnn(out, (hidden, cell))
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden, cell

    def init_hidden(self, batch_size):
        hidden = torch.zeros(1, batch_size, self.rnn_hidden_size).to("cuda")
        cell = torch.zeros(1, batch_size, self.rnn_hidden_size).to("cuda")
        return hidden, cell

In [44]:
vocab_size = len(char_array)
EMBED_DIM, RNN_HIDDEN_SIZE = 256, 512

model = RNN(vocab_size, EMBED_DIM, RNN_HIDDEN_SIZE).to("cuda")
model

RNN(
  (embedding): Embedding(80, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=80, bias=True)
)

In [45]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
NUM_EPOCHS = 1000
for e in range(NUM_EPOCHS):
    hidden, cell = model.init_hidden(BATCH_SIZE)
    for seq_batch, target_batch in seq_dl:
        # seq_batch, target_batch = seq_batch.to("cuda"), target_batch("cuda")
        optimizer.zero_grad()
        loss = 0
        for c in range(SEQ_LENGTH):
            pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
            hidden, cell = hidden.to(device), cell.to(device)
            loss += loss_fn(pred, target_batch[:, c])
        loss.backward()
        optimizer.step()
        loss = loss.item() / SEQ_LENGTH
        if e % 50:
            print(f"Epoch {e} | Loss: {loss:.4f}")

TypeError: 'Tensor' object is not callable

In [49]:
# NUM_EPOCHS = 1000
# for e in range(NUM_EPOCHS):
#     hidden, cell = model.init_hidden(BATCH_SIZE)
#     # seq_batch, target_batch = next(iter(seq_dl))
#     # seq_batch, target_batch = seq_batch.to("cuda"), target_batch("cuda")
#     optimizer.zero_grad()
#     loss = 0
#     for c in range(SEQ_LENGTH):
#         pred, hidden, cell = model(seq_batch[:, c], hidden, cell)

#         loss += loss_fn(pred, target_batch[:, c])
#     loss.backward()
#     optimizer.step()
#     loss = loss.item() / SEQ_LENGTH
#     if e % 50:
#         print(f"Epoch {e} | Loss: {loss:.4f}")